# AI Agent with Web Search and LiteLLM

## Introduction

This notebook demonstrates a comprehensive AI agent system that provides users with full control over model selection and processing approaches. The agent supports multiple AI models including Perplexity's Sonar models for web-enhanced responses and various LLM models for general processing.

### Key Features

- **Multi-Model Support**: Access to both Sonar (web-enabled) and regular LLM models
- **Flexible Processing Approaches**: Choose between Sonar-only, LLM-only, hybrid, or auto modes
- **User Preference Management**: Customizable default settings for seamless operation
- **Interactive Interface**: Command-based interaction system for easy model switching

### Processing Approaches

1. **Sonar-only**: Uses web-enhanced models for current information and research
2. **LLM-only**: Uses traditional language models without web access
3. **Hybrid**: Combines Sonar web search with LLM processing for comprehensive responses
4. **Auto**: Automatically selects the best approach based on query characteristics


To enhance the functionality of the CoreAI  environment, we need to install some libraries not pre-installed but required for this notebook. 

## Pre-requisites
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment:

```bash
export PROJECT_NAME="Agent"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}-myvenv --display-name="Python (${PROJECT_NAME}-myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}-myvenv)"

In [ ]:
import os

def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)

set_env_with_cache_dir("PIP_CACHE_DIR", "pip")

## Dependency Installation

Handles the installation of all required Python packages from the requirements.txt file using the virtual environment. The installation includes essential libraries like LiteLLM for unified model access, web scraping tools, and environment management utilities needed for the AI agent functionality.

In [ ]:
!. ./myvenv/bin/activate; pip install -r requirements.txt

## Environment Configuration

By default, the notebook will hide dot files (i.e. `.env`).
We propose to create a symbolic link to a local file that can be edited.

In a terminal, run `cp env.exmple env; ln -s env .env` then edit the `env` file to match your actual values for LiteLLM's API key and URL:
1. Add your LiteLLM proxy API key
2. Add your LiteLLM proxy base URL

**Security Note**: Never commit your actual API keys to version control.

In [ ]:
# Load environment configuration file
# This sets up the basic structure for API credentials

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
if 'LITELLM_PROXY_API_KEY' not in os.environ or 'LITELLM_PROXY_API_BASE' not in os.environ:
    print("Missing values -- FIX BEFORE CONTINUING")
print("✓ .env loaded")

## Import Required Libraries
Imports all necessary Python libraries and modules required for the AI agent functionality. The imports include web requests handling, type definitions, environment variable management, datetime utilities, JSON processing, and the core LiteLLM library for model interactions.

In [ ]:
# Import all required libraries
import os
import requests
from typing import List, Dict, Optional, Union
from dotenv import load_dotenv
import datetime
import json
import litellm
from litellm import completion

# Load and validate environment variables
load_dotenv()
LITELLM_PROXY_API_KEY = os.getenv("LITELLM_PROXY_API_KEY")
LITELLM_PROXY_API_BASE = os.getenv("LITELLM_PROXY_API_BASE")

if not LITELLM_PROXY_API_KEY or not LITELLM_PROXY_API_BASE:
    raise ValueError("Set LITELLM_PROXY_API_KEY and LITELLM_PROXY_API_BASE in your .env file.")


## UserChoiceAgent Class - Core Structure
Defines the main UserChoiceAgent class with its initialization method and core data structures. The class serves as the central hub for managing multiple AI models, user preferences, and conversation history, providing a unified interface for different processing approaches.

In [ ]:
class UserChoiceAgent:
    """
    A comprehensive AI agent that provides users with full control over model selection
    and processing approaches. Supports both web-enabled Sonar models and regular LLMs.
    """
    
    def __init__(self):
        """Initialize the agent with model discovery and user preference setup."""
        # Initialize model storage
        self.available_models = []
        self.sonar_models = []
        self.regular_models = []
        self.conversation_history = []
        
        # Default user preferences
        self.user_preferences = {
            "default_sonar_model": None,
            "default_llm_model": None,
            "preferred_approach": "hybrid"
        }
        
        # Initialize the agent
        self.load_available_models()
        self.setup_user_preferences()
        print(f" User Choice Agent initialized with {len(self.available_models)} models")


## Model Loading Methods
Implements the model discovery and categorization functionality that connects to the LiteLLM proxy. The method automatically fetches available models, separates Sonar (web-enabled) models from regular LLMs, and provides detailed information about each model type for user selection.

In [ ]:
def load_available_models(self):
    """Load and categorize available models from the LiteLLM proxy."""
    try:
        headers = {"Authorization": f"Bearer {LITELLM_PROXY_API_KEY}"}
        response = requests.get(f"{LITELLM_PROXY_API_BASE.rstrip('/')}/v1/models", headers=headers)
        
        if response.ok:
            models_data = response.json().get("data", [])
            self.available_models = [model.get("id") for model in models_data]
            
            # Categorize models based on 'sonar' keyword
            self.sonar_models = [model for model in self.available_models if 'sonar' in model.lower()]
            self.regular_models = [model for model in self.available_models if 'sonar' not in model.lower()]
            
            print(f" Found {len(self.sonar_models)} Sonar models: {', '.join(self.sonar_models)}")
            print(f" Found {len(self.regular_models)} Regular models: {', '.join(self.regular_models)}")
        else:
            print(f" Could not load models: {response.text}")
    except Exception as e:
        print(f" Error loading models: {e}")

# Add method to UserChoiceAgent class
UserChoiceAgent.load_available_models = load_available_models


## User Preference Setup
Provides an interactive interface for users to configure their default model preferences and processing approaches. The setup process guides users through selecting their preferred Sonar model, LLM model, and default processing strategy, ensuring a personalized experience tailored to their specific needs.

In [ ]:
def setup_user_preferences(self):
    """
    Interactive setup for user model preferences.
    Allows users to choose default Sonar model, LLM model, and processing approach.
    """
    print("\n SETTING UP YOUR MODEL PREFERENCES")
    print("=" * 50)
    
    # Choose default Sonar model
    if self.sonar_models:
        print(f"\n Available Sonar models for web search:")
        for i, model in enumerate(self.sonar_models, 1):
            print(f"{i}. {model}")
        
        while True:
            try:
                choice = input(f"\nChoose your preferred Sonar model (1-{len(self.sonar_models)}) or press Enter for first: ").strip()
                if not choice:
                    self.user_preferences["default_sonar_model"] = self.sonar_models[0]
                    break
                index = int(choice) - 1
                if 0 <= index < len(self.sonar_models):
                    self.user_preferences["default_sonar_model"] = self.sonar_models[index]
                    break
                else:
                    print("Invalid choice. Please try again.")
            except ValueError:
                print("Please enter a number or press Enter.")
    
    # Choose default LLM model
    if self.regular_models:
        print(f"\n Available LLM models for processing:")
        for i, model in enumerate(self.regular_models, 1):
            print(f"{i}. {model}")
        
        while True:
            try:
                choice = input(f"\nChoose your preferred LLM model (1-{len(self.regular_models)}) or press Enter for first: ").strip()
                if not choice:
                    self.user_preferences["default_llm_model"] = self.regular_models[0]
                    break
                index = int(choice) - 1
                if 0 <= index < len(self.regular_models):
                    self.user_preferences["default_llm_model"] = self.regular_models[index]
                    break
                else:
                    print("Invalid choice. Please try again.")
            except ValueError:
                print("Please enter a number or press Enter.")
    
    # Choose default processing approach
    approaches = ["sonar_only", "llm_only", "hybrid", "auto"]
    print(f"\n Default processing approaches:")
    for i, approach in enumerate(approaches, 1):
        descriptions = {
            "sonar_only": "Always use Sonar for web-enhanced responses",
            "llm_only": "Always use regular LLM without web search",
            "hybrid": "Use Sonar for search + LLM for processing",
            "auto": "Automatically choose best approach based on query"
        }
        print(f"{i}. {approach}: {descriptions[approach]}")
    
    while True:
        try:
            choice = input(f"\nChoose your preferred approach (1-4) or press Enter for hybrid: ").strip()
            if not choice:
                self.user_preferences["preferred_approach"] = "hybrid"
                break
            index = int(choice) - 1
            if 0 <= index < len(approaches):
                self.user_preferences["preferred_approach"] = approaches[index]
                break
            else:
                print("Invalid choice. Please try again.")
        except ValueError:
            print("Please enter a number or press Enter.")
    
    # Display final preferences
    print(f"\n Preferences set:")
    print(f"  Default Sonar: {self.user_preferences['default_sonar_model']}")
    print(f"  Default LLM: {self.user_preferences['default_llm_model']}")
    print(f"  Default Approach: {self.user_preferences['preferred_approach']}")

# Add method to UserChoiceAgent class
UserChoiceAgent.setup_user_preferences = setup_user_preferences


##  Response Generation Methods
Implements the core response generation functionality for individual model interactions. The method handles the communication with LiteLLM proxy, manages API calls, processes responses, and provides comprehensive error handling for robust model interactions.

In [ ]:
def generate_response(self, prompt: str, model: str) -> Dict:
    """Generate response using specified model."""
    try:
        print(f" Using {model}...")
        
        response = completion(
            model=f"litellm_proxy/{model}",
            messages=[{"role": "user", "content": prompt}],
            api_base=LITELLM_PROXY_API_BASE,
            api_key=LITELLM_PROXY_API_KEY,
            stream=False
        )
        
        return {
            "content": response.choices[0].message.content,
            "model": model,
            "type": "llm_response"
        }
        
    except Exception as e:
        return {"error": f"Error with {model}: {str(e)}"}

# Add method to UserChoiceAgent class
UserChoiceAgent.generate_response = generate_response


## Sonar Search Methods
Specializes in web-enabled search functionality using Perplexity's Sonar models. The implementation provides access to current web information, real-time data, and up-to-date content by leveraging Sonar's web search capabilities for enhanced response accuracy.



In [ ]:
def sonar_search(self, query: str, sonar_model: str = None) -> Dict:
    """Use Sonar model for web-enabled search and response generation."""
    if not sonar_model:
        sonar_model = self.user_preferences["default_sonar_model"]
    
    if not sonar_model:
        return {"error": "No Sonar model available"}
    
    try:
        print(f" Using {sonar_model} for web search...")
        
        response = completion(
            model=f"litellm_proxy/{sonar_model}",
            messages=[{"role": "user", "content": query}],
            api_base=LITELLM_PROXY_API_BASE,
            api_key=LITELLM_PROXY_API_KEY,
            stream=False
        )
        
        return {
            "content": response.choices[0].message.content,
            "model": sonar_model,
            "type": "sonar_search"
        }
        
    except Exception as e:
        return {"error": f"Sonar search failed: {str(e)}"}

# Add method to UserChoiceAgent class
UserChoiceAgent.sonar_search = sonar_search


## Hybrid Processing Methods
Implements the hybrid approach that combines web search with LLM processing for comprehensive responses. The method first gathers current information using Sonar models, then enhances and structures the content using regular LLMs for optimal readability and analysis.

In [ ]:
def hybrid_approach(self, query: str, sonar_model: str = None, llm_model: str = None) -> Dict:
    """Combine Sonar web search with LLM processing for comprehensive responses."""
    sonar_model = sonar_model or self.user_preferences["default_sonar_model"]
    llm_model = llm_model or self.user_preferences["default_llm_model"]
    
    if not sonar_model or not llm_model:
        return {"error": "Both Sonar and LLM models required for hybrid mode"}
    
    try:
        print(f" Hybrid Mode: {sonar_model} + {llm_model}")
        
        # Step 1: Get web-grounded information from Sonar
        sonar_result = self.sonar_search(query, sonar_model)
        if "error" in sonar_result:
            return sonar_result
        
        # Step 2: Use LLM to process and enhance the information
        enhancement_prompt = f"""
Based on the following web-grounded information, provide a comprehensive and well-structured answer to the query: "{query}"

Web-grounded information:
{sonar_result['content']}

Please:
1. Organize the information clearly
2. Add relevant context or explanations where helpful
3. Maintain accuracy to the source information
4. Structure the response for better readability
"""
        
        llm_result = self.generate_response(enhancement_prompt, llm_model)
        if "error" in llm_result:
            return llm_result
        
        return {
            "content": llm_result['content'],
            "sonar_content": sonar_result['content'],
            "sonar_model": sonar_model,
            "llm_model": llm_model,
            "type": "hybrid_response"
        }
        
    except Exception as e:
        return {"error": f"Hybrid processing failed: {str(e)}"}

# Add method to UserChoiceAgent class
UserChoiceAgent.hybrid_approach = hybrid_approach


## Query Processing Methods
Contains the main query processing logic that routes user requests to appropriate processing approaches. The method supports automatic approach detection, manual selection, and switching between different processing modes based on query characteristics and user preferences.

In [ ]:
def process_query(self, query: str, options: Dict = None) -> Dict:
    """Process query with full user choice options."""
    if not options:
        options = {}
    
    # Use user preferences as defaults
    approach = options.get("approach", self.user_preferences["preferred_approach"])
    sonar_model = options.get("sonar_model", self.user_preferences["default_sonar_model"])
    llm_model = options.get("llm_model", self.user_preferences["default_llm_model"])
    
    # Auto-detect approach if needed
    if approach == "auto":
        search_keywords = ['latest', 'current', 'recent', 'news', 'today', 'update', 'what is happening']
        if any(word in query.lower() for word in search_keywords):
            approach = "sonar_only"
        else:
            approach = "hybrid"
    
    # Execute based on approach
    if approach == "sonar_only":
        result = self.sonar_search(query, sonar_model)
    elif approach == "llm_only":
        result = self.generate_response(query, llm_model)
    elif approach == "hybrid":
        result = self.hybrid_approach(query, sonar_model, llm_model)
    else:
        result = {"error": f"Unknown approach: {approach}"}
    
    # Add to conversation history
    self.conversation_history.append({
        "query": query,
        "result": result,
        "options": options,
        "timestamp": datetime.datetime.now().isoformat()
    })
    
    return result

# Add method to UserChoiceAgent class
UserChoiceAgent.process_query = process_query


## Utility Methods
Provides essential utility functions for preference management and model information retrieval.

In [ ]:
def update_preferences(self, **kwargs):
    """Update user preferences programmatically."""
    for key, value in kwargs.items():
        if key in self.user_preferences:
            self.user_preferences[key] = value
            print(f"✓ Updated {key} to {value}")

def list_models(self) -> Dict[str, List[str]]:
    """Return categorized list of available models and current preferences."""
    return {
        "sonar_models": self.sonar_models,
        "regular_models": self.regular_models,
        "all_models": self.available_models,
        "user_preferences": self.user_preferences
    }

# Add methods to UserChoiceAgent class
UserChoiceAgent.update_preferences = update_preferences
UserChoiceAgent.list_models = list_models


## Agent Initialization

In [ ]:
# Initialize the agent with user choice
agent = UserChoiceAgent()

## Interactive Interface Function
Implements the comprehensive command-line interface that provides users with full control over the AI agent. The interface supports multiple command types, model switching, preference updates, and various query processing modes.

In [ ]:
def interactive_user_choice_mode():
    """
    Interactive mode with complete user control over model selection.
    Provides a command-line interface for various AI agent operations.
    """
    print("=" * 70)
    print(" USER CHOICE AGENT")
    print("=" * 70)
    print("Available Commands:")
    print("1. 'models' - List all available models and current preferences")
    print("2. 'prefs' - Update your model preferences")
    print("3. 'sonar <model>: <query>' - Use specific Sonar model")
    print("4. 'llm <model>: <query>' - Use specific LLM model")
    print("5. 'hybrid <sonar> + <llm>: <query>' - Custom hybrid")
    print("6. 'approach <type>: <query>' - Use specific approach (sonar_only/llm_only/hybrid/auto)")
    print("7. '<query>' - Use your default preferences")
    print("8. 'quit' - Exit")
    print("=" * 70)
    
    while True:
        try:
            user_input = input(f"\n Query: ").strip()
            
            # Handle quit command
            if user_input.lower() == 'quit':
                print(" Goodbye!")
                break
            
            # Handle models command - display available models and preferences
            elif user_input.lower() == 'models':
                models = agent.list_models()
                print(f"\n AVAILABLE MODELS:")
                print(f" Sonar Models: {', '.join(models['sonar_models'])}")
                print(f" Regular Models: {', '.join(models['regular_models'])}")
                print(f"\n YOUR CURRENT PREFERENCES:")
                for key, value in models['user_preferences'].items():
                    print(f"  {key}: {value}")
            
            # Handle preferences update
            elif user_input.lower() == 'prefs':
                print("\n UPDATE PREFERENCES:")
                print("1. Update default Sonar model")
                print("2. Update default LLM model") 
                print("3. Update default approach")
                
                choice = input("Choose option (1-3): ").strip()
                
                # Update Sonar model preference
                if choice == '1' and agent.sonar_models:
                    print("Available Sonar models:")
                    for i, model in enumerate(agent.sonar_models, 1):
                        print(f"{i}. {model}")
                    model_choice = input("Choose model number: ").strip()
                    try:
                        index = int(model_choice) - 1
                        if 0 <= index < len(agent.sonar_models):
                            agent.update_preferences(default_sonar_model=agent.sonar_models[index])
                    except ValueError:
                        print("Invalid choice")
                
                # Update LLM model preference
                elif choice == '2' and agent.regular_models:
                    print("Available LLM models:")
                    for i, model in enumerate(agent.regular_models, 1):
                        print(f"{i}. {model}")
                    model_choice = input("Choose model number: ").strip()
                    try:
                        index = int(model_choice) - 1
                        if 0 <= index < len(agent.regular_models):
                            agent.update_preferences(default_llm_model=agent.regular_models[index])
                    except ValueError:
                        print("Invalid choice")
                
                # Update approach preference
                elif choice == '3':
                    approaches = ["sonar_only", "llm_only", "hybrid", "auto"]
                    print("Available approaches:")
                    for i, approach in enumerate(approaches, 1):
                        print(f"{i}. {approach}")
                    approach_choice = input("Choose approach number: ").strip()
                    try:
                        index = int(approach_choice) - 1
                        if 0 <= index < len(approaches):
                            agent.update_preferences(preferred_approach=approaches[index])
                    except ValueError:
                        print("Invalid choice")
            
            # Handle specific Sonar model usage
            elif user_input.startswith('sonar ') and ': ' in user_input:
                parts = user_input[6:].split(': ', 1)
                model, query = parts[0].strip(), parts[1].strip()
                if model in agent.sonar_models:
                    result = agent.process_query(query, {"approach": "sonar_only", "sonar_model": model})
                    print(f"\n {model} Response:\n{result.get('content', result.get('error', 'No response'))}")
                else:
                    print(f" {model} not found in available Sonar models")
            
            # Handle specific LLM model usage
            elif user_input.startswith('llm ') and ': ' in user_input:
                parts = user_input[4:].split(': ', 1)
                model, query = parts[0].strip(), parts[1].strip()
                if model in agent.regular_models:
                    result = agent.process_query(query, {"approach": "llm_only", "llm_model": model})
                    print(f"\n {model} Response:\n{result.get('content', result.get('error', 'No response'))}")
                else:
                    print(f" {model} not found in available LLM models")
            
            # Handle hybrid approach with custom models
            elif user_input.startswith('hybrid ') and ': ' in user_input:
                parts = user_input[7:].split(': ', 1)
                models_part, query = parts[0].strip(), parts[1].strip()
                if ' + ' in models_part:
                    sonar_model, llm_model = models_part.split(' + ', 1)
                    sonar_model, llm_model = sonar_model.strip(), llm_model.strip()
                    result = agent.process_query(query, {
                        "approach": "hybrid",
                        "sonar_model": sonar_model,
                        "llm_model": llm_model
                    })
                    if result.get("type") == "hybrid_response":
                        print(f"\n Hybrid ({sonar_model} + {llm_model}):\n{result['content']}")
                    else:
                        print(f"\n Error: {result.get('error', 'Unknown error')}")
                else:
                    print(" Format: hybrid <sonar_model> + <llm_model>: <query>")
            
            # Handle specific approach usage
            elif user_input.startswith('approach ') and ': ' in user_input:
                parts = user_input[9:].split(': ', 1)
                approach, query = parts[0].strip(), parts[1].strip()
                if approach in ["sonar_only", "llm_only", "hybrid", "auto"]:
                    result = agent.process_query(query, {"approach": approach})
                    print(f"\n {approach.upper()} Response:\n{result.get('content', result.get('error', 'No response'))}")
                else:
                    print(" Valid approaches: sonar_only, llm_only, hybrid, auto")
            
            # Handle default query processing
            else:
                # Use default preferences for processing
                result = agent.process_query(user_input)
                print(f"\n Response:\n{result.get('content', result.get('error', 'No response'))}")
            
        except KeyboardInterrupt:
            print("\n Goodbye!")
            break
        except Exception as e:
            print(f" Error: {e}")

# Display usage information
print(" User Choice Agent loaded!")
print(" Run interactive_user_choice_mode() to start!")


## Start Interactive Mode

In [ ]:
# Start the interactive AI agent interface
interactive_user_choice_mode()